# Caso 6 — Análise por Consulta

Selecionamos, a partir das diferenças de AP calculadas no Caso 5:

- **2 consultas em que o BM25 é claramente superior** ao Modelo Vetorial;
- **2 consultas em que o Modelo Vetorial é claramente superior** ao BM25;
- **2 consultas em que ambos os modelos têm desempenho insatisfatório**.

Para cada uma, mostramos os 5 primeiros documentos retornados por cada
modelo, indicando quais são de fato relevantes segundo o qrels.


In [ ]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pickle
import pandas as pd
from IPython.display import display

from src.cranfield_data import load_cranfield

df_docs, df_queries, df_qrels = load_cranfield()
doc_title = dict(zip(df_docs.doc_id, df_docs.title.str.replace("\n", " ")))
query_text = dict(zip(df_queries.query_id, df_queries.text.str.replace("\n", " ")))

RESULTS_DIR = project_root / "data" / "processed"
vsm_eval = pd.read_csv(RESULTS_DIR / "vsm_perquery.csv", dtype={"query_id": str}).set_index("query_id")
bm25_eval = pd.read_csv(RESULTS_DIR / "bm25_perquery.csv", dtype={"query_id": str}).set_index("query_id")
with open(RESULTS_DIR / "vsm_rankings.pkl", "rb") as f:
    vsm_rankings = pickle.load(f)
with open(RESULTS_DIR / "bm25_rankings.pkl", "rb") as f:
    bm25_rankings = pickle.load(f)


def show_top_n(qid, n=5):
    """Mostra os top-n documentos de cada modelo para uma consulta, com o
    grau de relevância (qrels) de cada documento retornado."""
    grades = dict(zip(df_qrels[df_qrels.query_id == qid].doc_id,
                       df_qrels[df_qrels.query_id == qid].relevance))
    print(f"Consulta {qid}: {query_text[qid]}")
    print(f"AP -> BM25: {bm25_eval.loc[qid, 'AP']:.3f} | "
          f"Modelo Vetorial: {vsm_eval.loc[qid, 'AP']:.3f}")
    frames = {}
    for nome, rankings in [("BM25", bm25_rankings), ("Modelo Vetorial", vsm_rankings)]:
        rows = []
        for rank, did in enumerate(rankings[qid][:n], start=1):
            grade = grades.get(did)
            rows.append({
                "rank": rank,
                "doc_id": did,
                "grau": grade if grade is not None else "não julgado",
                "relevante?": "SIM" if (grade is not None and grade >= 1) else "não",
                "titulo": doc_title[did][:75],
            })
        frames[nome] = pd.DataFrame(rows).set_index("rank")
    for nome, frame in frames.items():
        print(f"-- Top-{n} {nome} --")
        display(frame)
    print()

## 1) Consultas em que o BM25 é claramente superior

**Consulta 167** (AP: BM25 1,000 vs. Vetorial 0,098): há dois documentos
relevantes (274, grau 2, e 82, grau 3). O Modelo Vetorial não traz
nenhum dos dois ao Top-5 — eles só aparecem nas posições 14ª e 16ª do
ranking completo — porque o topo é dominado por documentos não julgados
como o 553, que repete fortemente o termo `ablat` num texto curto (alta
concentração lexical, alto peso tf-idf). O BM25, graças à saturação de
frequência de termo (parâmetro k1) e à normalização por tamanho de
documento, não deixa esse tipo de repetição dominar o score e recupera
os dois documentos relevantes exatamente nas posições 1 e 2, obtendo AP
perfeito.

**Consulta 173** (AP: BM25 1,000 vs. Vetorial 0,583): os dois documentos
relevantes (367 e 451, sobre o "método de Lyapunov") existem, mas o
Modelo Vetorial insere na 1ª posição o documento 532 — não relevante
(grau -1) — que também menciona "estabilidade" e "segundo método", por
coincidência lexical. O BM25 ainda recupera 532 na 3ª posição (não é
imune ao mesmo problema), mas pondera menos essa coincidência e consegue
colocar os dois documentos relevantes exatamente nas posições 1 e 2,
obtendo AP perfeito.


In [ ]:
for qid in ["167", "173"]:
    show_top_n(qid)

Consulta 167: exact solution methods for calculating the ablative mass loss of a material ablating at high temperatures in a hypersonic flight environment .
AP -> BM25: 1.000 | Modelo Vetorial: 0.098
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,274,2,SIM,analysis of quartz and teflon shields for a pa...
2,82,3,SIM,theoretical investigation of the ablation of a...
3,553,não julgado,não,ablation of glassy materials around blunt bodi...
4,1279,não julgado,não,sublimation in a hypersonic environment .
5,1043,não julgado,não,"on transverse vibrations of thin, shallow elas..."


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,553,não julgado,não,ablation of glassy materials around blunt bodi...
2,1279,não julgado,não,sublimation in a hypersonic environment .
3,1099,não julgado,não,a theoretical study of stagnation point ablati...
4,1097,não julgado,não,experimental ablation cooling .
5,1100,não julgado,não,an analytical investigation of ablation .



Consulta 173: references on lyapunov's method on the stability of linear differential equations with periodic coefficients .
AP -> BM25: 1.000 | Modelo Vetorial: 0.583
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,367,1,SIM,control system and analysis and design via the...
2,451,1,SIM,liapunov's methods in automatic control theory .
3,532,-1,não,pitch-yaw stability of a missile oscillating i...
4,917,não julgado,não,a method of calculating the short period longi...
5,767,não julgado,não,mathematical techniques applying to the therma...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,532,-1,não,pitch-yaw stability of a missile oscillating i...
2,367,1,SIM,control system and analysis and design via the...
3,451,1,SIM,liapunov's methods in automatic control theory .
4,917,não julgado,não,a method of calculating the short period longi...
5,1073,não julgado,não,a practical method for numerical evaluation of...


## 2) Consultas em que o Modelo Vetorial é claramente superior

**Consulta 113** (AP: Vetorial 0,287 vs. BM25 0,179): há 4 documentos
relevantes julgados para esta consulta (746, 748, 749, 265), mas a
maioria está profundamente enterrada em ambos os rankings (posições
17–440) — a diferença de AP vem quase toda da posição do documento mais
facilmente recuperável, 748 (grau 3): o Modelo Vetorial o coloca em 1º
lugar, enquanto o BM25 o coloca em 2º, atrás do documento não julgado
704. Como o AP pondera fortemente os acertos nas primeiras posições,
essa diferença de apenas uma posição já é suficiente para o Modelo
Vetorial superar o BM25 nesta consulta.

**Consulta 148** (AP: Vetorial 0,450 vs. BM25 0,239): padrão semelhante
— os documentos relevantes 1048 e 1050 aparecem nas 2 primeiras posições
do Modelo Vetorial, mas só nas posições 2 e 3 do BM25, que insere antes
(1ª posição) o documento 956 (não relevante, grau -1) sobre um tema
visualmente muito próximo ("cilindros sanduíche corrugados").


In [ ]:
for qid in ["113", "148"]:
    show_top_n(qid)

Consulta 113: what data exists on oscillatory aerodynamic forces on control surfaces at transonic mach numbers .
AP -> BM25: 0.179 | Modelo Vetorial: 0.287
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,704,não julgado,não,a systematic kernel function procedure for det...
2,748,3,SIM,subsonic aerodynamic flutter derivatives for w...
3,815,não julgado,não,investigation of several blunt bodies to deter...
4,709,não julgado,não,static longitudinal aerodynamic characteristic...
5,708,não julgado,não,aerodynamic characteristics of two winged reen...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,748,3,SIM,subsonic aerodynamic flutter derivatives for w...
2,1272,não julgado,não,oscillatory aerodynamic coefficients for a uni...
3,716,não julgado,não,study of the oscillatory motion of manned vehi...
4,801,não julgado,não,experimental study of the equivalence of trans...
5,1290,não julgado,não,measured and calculated subsonic and transonic...



Consulta 148: papers on small deflection theory for buckling of sandwich cylinders .
AP -> BM25: 0.239 | Modelo Vetorial: 0.450
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,956,-1,não,elastic stability of simply supported corrugat...
2,1048,2,SIM,a small deflection theory for curved sandwich ...
3,1050,2,SIM,compressive buckling of simply supported curve...
4,1126,não julgado,não,an engineer's conceptual approach to the buckl...
5,928,não julgado,não,a new theory for the buckling of thin cylinder...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,1048,2,SIM,a small deflection theory for curved sandwich ...
2,1050,2,SIM,compressive buckling of simply supported curve...
3,761,não julgado,não,buckling of sandwich under normal pressure .
4,956,-1,não,elastic stability of simply supported corrugat...
5,720,não julgado,não,a note on the use of sandwich structures in se...


## 3) Consultas em que ambos os modelos falham

**Consulta 31** ("que tamanho de placa de extremidade pode ser usado com
segurança para simular condições de escoamento bidimensional..."): possui
um único documento relevante (776, grau 4), que **nenhum dos dois modelos
coloca nem perto do Top-10** — ele aparece na posição 1301 (de 1400) em
**ambos** os rankings, ou seja, os dois modelos concordam ao classificá-lo
como quase irrelevante. Pior ainda: o documento 751 (explicitamente
**não relevante**, grau -1) — que tem forte sobreposição lexical com a
consulta ("end plates", "three dimensional flow") mas trata do problema
oposto (como *evitar* o efeito tridimensional, não como simulá-lo) — é
colocado em 1º lugar por **ambos** os modelos.

**Consulta 22** ("alguém mais descobriu que o atrito de pele turbulento
não é muito sensível à variação da viscosidade com a temperatura..."): o
único documento relevante (68, grau 1) fica na posição 584 em **ambos**
os rankings. O título do documento relevante ("some aspects of
air-helium simulation and hypersonic approximations") **não compartilha
praticamente nenhum termo de conteúdo** com a consulta — a relevância
aparentemente depende de uma ligação conceitual (mesma técnica
experimental discutida no corpo do texto) que não aparece no título nem
é capturada por nenhum modelo puramente lexical.

Em ambos os casos o problema não está em qual modelo foi escolhido — é o
clássico **problema de vocabulário** (sinonímia/paráfrase): quando a
consulta e o documento relevante descrevem o mesmo conceito com palavras
de superfície muito diferentes, nenhum modelo baseado em correspondência
exata de termos (bag-of-words) consegue recuperá-lo bem. Isso é retomado
com mais profundidade no Caso 9 (Análise de Erros).


In [ ]:
for qid in ["31", "22"]:
    show_top_n(qid)

Consulta 31: what size of end plate can be safely used to simulate two-dimensional flow conditions over a bluff cylindrical body of finite aspect ratio .
AP -> BM25: 0.001 | Modelo Vetorial: 0.001
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,751,-1,não,a note on the use of end plates to prevent thr...
2,1209,não julgado,não,aerodynamic processes in the downwash-impingem...
3,1153,não julgado,não,a study of the simulation of flow with free st...
4,918,não julgado,não,on the low aspect ratio oscillating rectangula...
5,895,não julgado,não,the airforces on the low aspect ratio rectangu...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,751,-1,não,a note on the use of end plates to prevent thr...
2,68,não julgado,não,some aspects of air-helium simulation and hype...
3,918,não julgado,não,on the low aspect ratio oscillating rectangula...
4,247,não julgado,não,the calculation of the pressure distribution o...
5,916,não julgado,não,the flow around oscillating low aspect ratio w...



Consulta 22: did anyone else discover that the turbulent skin friction is not over sensitive to the nature of the variation of the viscosity with temperature .
AP -> BM25: 0.002 | Modelo Vetorial: 0.002
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,125,não julgado,não,measurements of skin friction of the compressi...
2,560,não julgado,não,a theoretical study of the effect of upstream ...
3,413,não julgado,não,turbulent skin friction at high mach numbers a...
4,254,não julgado,não,boundary layers with suction and injection . a...
5,307,não julgado,não,an approximate solution of hypersonic laminar ...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,125,não julgado,não,measurements of skin friction of the compressi...
2,254,não julgado,não,boundary layers with suction and injection . a...
3,413,não julgado,não,turbulent skin friction at high mach numbers a...
4,560,não julgado,não,a theoretical study of the effect of upstream ...
5,346,não julgado,não,measurements of turbulent friction on a smooth...
